# HUC04 Domain & Tiles Prep
Prepare the lookup grids/tiles to use for querying The National Map API for 3DEP 1m DEMs

In [1]:
import os
import sys
import pandas as pd
import geopandas as gpd

sys.path.append("/workspaces/ffrd-3dep/src")

from tnm.tnm import prep_domain, process_single_dem

grids_file = "/workspaces/ffrd-3dep/data/input/grids/10_km_cell_grid.parquet"
domain_file = "/workspaces/ffrd-3dep/data/input/domain/allegheny.geojson"
logs_file = "/workspaces/ffrd-3dep/data/output/logs/logs.csv"

BUFFER = 5000 # Buffer distance in feet
TARGET_CRS = """PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_FFRD",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",29.5],PARAMETER["Standard_Parallel_2",45.5],PARAMETER["Latitude_Of_Origin",23.0],UNIT["Foot",0.3048]]"""

# Intersect the Domain & TNM Grids

In [4]:
domain_gdf = prep_domain(domain_file, buffer_distance=BUFFER, target_crs=TARGET_CRS)
grids_gdf = gpd.read_parquet(grids_file)
grids_gdf = grids_gdf.to_crs(TARGET_CRS)
grids_gdf = gpd.sjoin(
    grids_gdf,
    domain_gdf,
    how="inner",
    predicate="intersects",
).drop(columns=["index_right"])
grids_gdf

,slug,name,utm_zone,cellgrid_id,geometry
117899,x64y471-8,x64y471,17,37fd9c17-526a-4227-9403-313b07592a8c,"POLYGON ((4456586.778 7479526.273, 4451458.063..."
117900,x65y471-8,x65y471,17,37fd9c17-526a-4227-9403-313b07592a8c,"POLYGON ((4488796.295 7484570.798, 4483672.174..."
117901,x66y471-8,x66y471,17,37fd9c17-526a-4227-9403-313b07592a8c,"POLYGON ((4521005.417 7489609.612, 4515885.901..."
117904,x69y471-8,x69y471,17,37fd9c17-526a-4227-9403-313b07592a8c,"POLYGON ((4617629.572 7504691.681, 4612523.916..."
117905,x70y471-8,x70y471,17,37fd9c17-526a-4227-9403-313b07592a8c,"POLYGON ((4649836.27 7509707.543, 4644735.25 7..."
...,...,...,...,...,...
134464,x25y464-9,x25y464,18,37fd9c17-526a-4227-9403-313b07592a8c,"POLYGON ((4842072.266 7305447.242, 4834738.799..."
134465,x26y464-9,x26y464,18,37fd9c17-526a-4227-9403-313b07592a8c,"POLYGON ((4873810.608 7312703.495, 4866480.385..."
134515,x24y463-9,x24y463,18,37fd9c17-526a-4227-9403-313b07592a8c,"POLYGON ((4817672.174 7265973.112, 4810337.075..."
134516,x25y463-9,x25y463,18,37fd9c17-526a-4227-9403-313b07592a8c,"POLYGON ((4849404.236 7273231.035, 4842072.266..."


In [5]:
m = grids_gdf.explore(name="Grids", style_kwds={"fillOpacity": 0.3})
domain_gdf.explore(m=m, name="Domain", color="red", style_kwds={"fillOpacity": 0.1})

# Process a Single Tile
Download the following tile as a test case: 

https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/1m/Projects/NY_Southwest_2_Co_2016/TIFF/USGS_one_meter_x24y466_NY_Southwest_2_Co_2016.tif

Place the downloaded tile in the notebooks folder

In [ ]:
test_output = "/workspaces/ffrd-3dep/notebooks/output"
test_tile = "/workspaces/ffrd-3dep/notebooks/USGS_one_meter_x24y466_NY_Southwest_2_Co_2016.tif"
TARGET_RES = 4
NODATA = -9999

os.makedirs(test_output, exist_ok=True)

args = (
    test_tile,
    f"{test_output}/test_output.tif",
    domain_gdf.geometry.iloc[0].wkb,
    domain_gdf.crs.to_wkt(),
    TARGET_CRS,
    TARGET_RES,
    NODATA,
    "LZW",
)

result = process_single_dem(args)
result

{'path': '/workspaces/ffrd-3dep/notebooks/USGS_one_meter_x24y466_NY_Southwest_2_Co_2016.tif',
 'output_path': '/workspaces/ffrd-3dep/notebooks/output/test_output.tif',
 'success': True,
 'message': 'Success'}

# Check Logs
After running the get_dem.py script check the logs to see if any failures occurred due to a misread

In [2]:
logs_df = pd.read_csv(logs_file)
logs_df

,path,output_path,success,message
0,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_one_...,False,Tile does not intersect with domain boundary
1,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,False,Tile does not intersect with domain boundary
2,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,False,Clip resulted in empty raster
3,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,False,Tile does not intersect with domain boundary
4,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,False,Clip resulted in empty raster
...,...,...,...,...
635,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,True,Success
636,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,False,Tile does not intersect with domain boundary
637,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,False,Tile does not intersect with domain boundary
638,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,False,Tile does not intersect with domain boundary


In [4]:
logs_df.message.unique()

<ArrowStringArray>
['Tile does not intersect with domain boundary',
                'Clip resulted in empty raster',
                                      'Success']
Length: 3, dtype: str

In [5]:
logs_df[logs_df.success == True]

,path,output_path,success,message
16,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_one_...,True,Success
26,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_one_...,True,Success
27,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_one_...,True,Success
31,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_one_...,True,Success
32,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_one_...,True,Success
...,...,...,...,...
631,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,True,Success
632,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,True,Success
633,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,True,Success
634,https://prd-tnm.s3.amazonaws.com/StagedProduct...,/workspaces/daa_apps/3DEP/Tiles_Test/USGS_1M_1...,True,Success
